In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

torch.manual_seed(42)

# Dummy dataset
X = torch.randn(1000, 10)

# Labels: 0 = Not Spam, 1 = Spam
y = torch.randint(0, 2, (1000, 1)).float()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

C:\Users\DeLL\AppData\Local\Temp\ipykernel_20616\4119608520.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
C:\Users\DeLL\AppData\Local\Temp\ipykernel_20616\4119608520.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
C:\Users\DeLL\AppData\Local\Temp\ipykernel_20616\4119608520.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.float32)
C:\Users\DeLL\AppData\Local\Temp\ipykernel_20616

In [3]:
class EmailClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

model = EmailClassifier()

In [4]:
loss_fn = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [5]:
epochs = 100

for epoch in range(epochs):

    # Training
    model.train()

    logits = model(X_train)

    loss = loss_fn(logits, y_train)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    # Evaluation
    model.eval()

    with torch.inference_mode():

        test_logits = model(X_test)

        test_loss = loss_fn(
            test_logits,
            y_test
        )

        probs = torch.sigmoid(test_logits)

        predictions = (probs >= 0.5).float()

        accuracy = (
            (predictions == y_test)
            .sum()
            .item()
            / len(y_test)
        ) * 100

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {loss:.4f} | "
            f"Test Loss: {test_loss:.4f} | "
            f"Accuracy: {accuracy:.2f}%"
        )

Epoch 000 | Train Loss: 0.6961 | Test Loss: 0.6934 | Accuracy: 51.50%
Epoch 010 | Train Loss: 0.6924 | Test Loss: 0.6916 | Accuracy: 53.50%
Epoch 020 | Train Loss: 0.6900 | Test Loss: 0.6905 | Accuracy: 56.50%
Epoch 030 | Train Loss: 0.6877 | Test Loss: 0.6898 | Accuracy: 56.00%
Epoch 040 | Train Loss: 0.6850 | Test Loss: 0.6888 | Accuracy: 54.50%
Epoch 050 | Train Loss: 0.6820 | Test Loss: 0.6881 | Accuracy: 56.00%
Epoch 060 | Train Loss: 0.6785 | Test Loss: 0.6881 | Accuracy: 55.00%
Epoch 070 | Train Loss: 0.6742 | Test Loss: 0.6885 | Accuracy: 57.50%
Epoch 080 | Train Loss: 0.6691 | Test Loss: 0.6903 | Accuracy: 57.50%
Epoch 090 | Train Loss: 0.6632 | Test Loss: 0.6934 | Accuracy: 58.00%


In [6]:
new_email = torch.tensor([
    [0.5, -1.2, 0.8, 1.4, -0.2,
     0.6, -0.5, 1.1, 0.3, 0.9]
], dtype=torch.float32)

model.eval()

with torch.inference_mode():

    logit = model(new_email)

    prob = torch.sigmoid(logit)

    pred = (prob >= 0.5).float()

print("Spam Probability:", prob.item())
print("Prediction:", "Spam" if pred.item() == 1 else "Not Spam")

Spam Probability: 0.42570263147354126
Prediction: Not Spam
